# GPU Architecture & the Execution Model

Companion notebook for the [GPU Architecture lesson](https://ml-viz.vercel.app/courses/gpu-programming/01-gpu-architecture).

We can't run real CUDA here, but the *principles* are just arithmetic. We model **throughput vs.
latency**, use **Amdahl's law** to see why a parallel machine needs a parallel workload, and
simulate **latency hiding** — the core trick that lets a GPU shrug off slow memory. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})

## 1 — Throughput vs. latency

A CPU finishes a single task quickly (low latency) but has few cores. A GPU has a high *per-task*
latency but enormous parallelism, so its **throughput** (tasks completed per unit time) dominates
once there's enough work. We model wall-clock time to finish `n` independent tasks.

In [ ]:
def finish_time(n_tasks, cores, latency_per_task):
    """Wall-clock time to finish n independent tasks on `cores` parallel lanes."""
    waves = np.ceil(n_tasks / cores)      # tasks run `cores` at a time
    return waves * latency_per_task

n = np.array([1, 8, 64, 512, 4096, 32768])
cpu = finish_time(n, cores=8,    latency_per_task=1.0)    # few fast cores
gpu = finish_time(n, cores=2048, latency_per_task=3.0)    # many slower cores

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.loglog(n, cpu, 'o-', color='#fb7185', label='CPU (8 cores, fast)')
ax.loglog(n, gpu, 's-', color='#2dd4bf', label='GPU (2048 cores, slower each)')
ax.set_xlabel('number of independent tasks'); ax.set_ylabel('wall-clock time (a.u.)')
ax.set_title('GPU wins by throughput once there is enough parallel work')
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

print(f"1 task:     CPU {cpu[0]:.0f}  vs GPU {gpu[0]:.0f}  -> CPU wins (latency)")
print(f"32768 tasks: CPU {cpu[-1]:.0f} vs GPU {gpu[-1]:.0f} -> GPU wins (throughput)")

## 2 — Amdahl's law: parallel hardware needs parallel work

If a fraction $p$ of a program is parallelizable, the maximum speedup with $s$ parallel lanes is
$$\text{speedup}(s) = \frac{1}{(1-p) + p/s}.$$
The serial remainder $(1-p)$ caps the benefit no matter how many cores you add — which is why
branchy, sequential code wastes a GPU.

In [ ]:
def amdahl(p, s):
    return 1.0 / ((1 - p) + p / s)

s = np.logspace(0, 4, 100)
fig, ax = plt.subplots(figsize=(8, 4.5))
for p, c in [(0.50, '#fb7185'), (0.90, '#eab308'), (0.99, '#818cf8'), (0.999, '#2dd4bf')]:
    ax.semilogx(s, amdahl(p, s), color=c, label=f'p = {p}')
ax.set_xlabel('parallel lanes (cores)'); ax.set_ylabel('speedup')
ax.set_title("Amdahl's law: the serial fraction caps GPU speedup")
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

print(f"p=0.99, infinite cores -> max speedup = {1/(1-0.99):.0f}x")
print(f"p=0.90, infinite cores -> max speedup = {1/(1-0.90):.0f}x  (a 10% serial part caps you at 10x)")

## 3 — Latency hiding with resident warps

A global-memory access costs hundreds of cycles. The GPU hides this by keeping many warps resident:
when one stalls on memory, the SM instantly runs another. We model an SM that, each cycle, runs any
warp that isn't currently waiting on memory, and measure utilization as we add warps.

In [ ]:
def sm_utilization(n_warps, mem_latency=200, compute_burst=10, cycles=5000):
    """Each warp computes for `compute_burst` cycles, then stalls `mem_latency` cycles, repeat.
    The SM runs one ready warp per cycle. Returns fraction of cycles the SM was busy."""
    ready_at = np.zeros(n_warps)        # cycle each warp becomes ready
    remaining = np.full(n_warps, compute_burst)
    busy = 0
    for c in range(cycles):
        ready = np.where(ready_at <= c)[0]
        if len(ready):
            w = ready[0]                # run one ready warp this cycle
            busy += 1
            remaining[w] -= 1
            if remaining[w] == 0:       # burst done -> stall on memory
                ready_at[w] = c + mem_latency
                remaining[w] = compute_burst
    return busy / cycles

warp_counts = [1, 2, 4, 8, 16, 24, 32, 48, 64]
util = [sm_utilization(w) for w in warp_counts]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(warp_counts, util, 'o-', color='#2dd4bf')
ax.axhline(1.0, ls='--', color='#555')
ax.set_xlabel('resident warps per SM'); ax.set_ylabel('SM utilization')
ax.set_title('More resident warps hide memory latency (until the SM saturates)')
ax.set_ylim(0, 1.05); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# With burst=10 and latency=200, you need ~ (10+200)/10 = 21 warps to fully hide latency.
print(f" 1 warp:  utilization = {util[0]:.2f}  (mostly stalled on memory)")
print(f"64 warps: utilization = {util[-1]:.2f}  (latency fully hidden)")

## ✏️ Your turn

**Exercise.** Implement `warps_to_hide_latency(mem_latency, compute_burst)` returning the *minimum*
number of warps needed to fully hide memory latency (keep the SM busy every cycle). From the lesson:
you need enough independent work in flight to cover the stall, i.e.
`ceil((compute_burst + mem_latency) / compute_burst)`.

In [ ]:
import math

def warps_to_hide_latency(mem_latency, compute_burst):
    # TODO(you): return the minimum warp count to keep the SM busy through the memory stall
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert warps_to_hide_latency(200, 10) == 21
assert warps_to_hide_latency(400, 10) == 41
assert warps_to_hide_latency(100, 100) == 2
# Cross-check against the simulation: at the predicted count, utilization should be ~1.0
w = warps_to_hide_latency(200, 10)
assert sm_utilization(w) > 0.95
print(f"\u2713 need {w} warps to hide a 200-cycle stall behind 10-cycle bursts")

<details>
<summary>Solution</summary>

```python
def warps_to_hide_latency(mem_latency, compute_burst):
    return math.ceil((compute_burst + mem_latency) / compute_burst)
```

This is Little's Law for the GPU: to keep the arithmetic units busy you need enough independent
work *in flight* to cover the round-trip latency. Occupancy (Lesson 3) is the hardware's name for
how many warps you actually keep resident, and it exists precisely to enable this hiding.

</details>